# Pixelle-Video · PDF → Video × Wan2GP on Google Colab

Turns a **PDF document** (research paper, report, book chapter, slide deck, manual...) into a narrated short video — **PDF → AI digest → AI script → AI images/videos → TTS narration → final short video** — with review/edit checkpoints between every stage:

```
① Ingest    PDF → cleaned text + metadata + TOC (PyMuPDF)
② Digest    map-reduce LLM analysis → core message · grounded key insights ·
            hook ideas · ONE coherent visual world · tone · language
③ Script    digest → video title + scene narrations  →  you edit / ✨AI-rewrite
④ Visuals   digest-aware media prompts (shared visual world)  →  you edit / 🎲regenerate
⑤ Scenes    per scene: 🎤 audio → 🖼️/🎬 image-or-video → 🎞️ segment
⑥ Final     compose all segments (+ optional BGM) → final.mp4
```

Two things make the PDF pipeline special:
- **Grounding** — every fact, number and quote in the script must come from the document (the digest keeps the evidence attached to each insight), so the video is faithful to the source.
- **One visual world** — the digest picks a single visual setting/motif that fits the document; every scene prompt lives inside it, so the video feels art-directed instead of stitched together.

Media is generated with the **Wan2GP in-process backend** (models loaded directly in this runtime through WanGP's Python API). No ComfyUI server, no RunningHub key.

Run the cells in order. Dependencies are installed **once** and shared by everything.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. The defaults below are sized for it: **Z-Image Turbo 6B** for images and **Wan 2.1 1.3B** for video clips. On a bigger GPU (L4/A100) switch to `wan2gp/image_qwen.json`, `wan2gp/video_wan2.1_fusionx.json` or `wan2gp/video_ltx2_distilled.json`.

> **LLM note:** Pixelle-Video needs an OpenAI-compatible LLM endpoint (Qwen/DashScope, DeepSeek, OpenAI, Ollama, your own vLLM tunnel, ...) to analyze the PDF and write the script and media prompts. Fill it in at step 6.

> **Scanned PDFs** (images without a text layer) extract almost no text — run OCR on them first (e.g. `ocrmypdf in.pdf out.pdf`) and upload the result.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime -> Change runtime type, select GPU, save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path

Choose where Wan2GP should be installed. It bundles all three Pixelle-Video flavors:
- `Pixelle_video/` — the original fully-automatic app (also holds the shared config / workflows / templates / output)
- `Pixelle_video_scene_by_scene/` — the step-by-step engine this pipeline builds on
- `Pixelle_video_pdf/` — the PDF → video engine used by this notebook


In [ ]:
from pathlib import Path

WAN2GP_ROOT  = Path('/content/wan2gp').resolve()
PIXELLE_ROOT = WAN2GP_ROOT / 'Pixelle_video'
SBS_ROOT     = WAN2GP_ROOT / 'Pixelle_video_scene_by_scene'
PDF_ROOT     = WAN2GP_ROOT / 'Pixelle_video_pdf'
print(f'Wan2GP will be installed to:   {WAN2GP_ROOT}')
print(f'Pixelle-Video (core + data):   {PIXELLE_ROOT}')
print(f'Scene-by-Scene engine:         {SBS_ROOT}')
print(f'PDF → Video app:               {PDF_ROOT}')


## 3. Download or update Wan2GP

Clone the repository on the `feature/pixelle-pdf` branch (the one that contains `Pixelle_video_pdf/`); pull the latest changes if it already exists.


In [ ]:
import subprocess

repo_url = 'https://github.com/hoangthvn2201/Wan2GP'
branch = 'feature/pixelle-pdf'   # change to 'main' once the branch is merged

if WAN2GP_ROOT.exists():
    print('Repository already exists. Updating...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', branch], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', 'origin', branch], check=True)


## 4. Install system dependencies

Shared libraries for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.


In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies (one-time)

A **single install** covers everything:

- `requirements.txt` (Wan2GP) — torch ecosystem, diffusers, loguru, pydantic, moviepy, ffmpeg-python, ...
- `Pixelle_video/requirements.txt` — the Pixelle extras (streamlit, openai, edge-tts, comfykit, playwright, ...)
- `Pixelle_video_pdf/requirements.txt` — PDF text extraction (PyMuPDF + pypdf fallback)

The scene-by-scene engine has no extra dependencies of its own. Afterwards Chromium is installed for the HTML frame-template rendering. This cell takes several minutes.


In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

# One-time install: Wan2GP requirements + Pixelle-Video extras + PDF extras in a single resolve
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    '-r', str(PIXELLE_ROOT / 'requirements.txt'),
    '-r', str(PDF_ROOT / 'requirements.txt')], check=True, env=env)

# Chromium for HTML frame template rendering (Pixelle composes subtitles via Playwright)
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True, env=env)

# Re-assert a modern setuptools AFTER all installs: the dependency resolve can
# remove it from /usr/local, letting Ubuntu's ancient system pkg_resources
# (which still uses pkgutil.ImpImporter, removed in Python 3.12) shadow it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend.


In [ ]:
from pathlib import Path

target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 5c. (Optional) Vietnamese TTS — VieNeu-TTS

**Skip this cell** unless you want on-device Vietnamese narration via [VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS).

It installs *on top of* the standard environment and does **not** change any package pinned in step 5 (`vieneu` itself is installed `--no-deps` — see `Pixelle_video/requirements-vieneu.txt` for why). Expect **~10–15 minutes**: `llama-cpp-python` compiles from source (cmake/gcc are preinstalled on Colab) and `sea-g2p` needs a Rust toolchain (installed automatically below). Model weights (GGUF + ONNX codec, ~1 GB) are downloaded from HuggingFace on the first synthesis.

Afterwards set `TTS_MODE = 'vieneu'` in step 6. Preset voices (fuzzy-matched): `Xuân Vĩnh (Nam - Miền Nam)`, `Phạm Tuyên (Nam - Miền Bắc)`, `Bích Ngọc (Nữ - Miền Bắc)`, `Thục Đoan (Nữ - Miền Nam)` — e.g. `TTS_VOICE = 'Bích Ngọc'` works.

> Tip: pair it with `NARRATION_LANGUAGE = 'Vietnamese'` in step 10 to make a Vietnamese video about an English document.


In [ ]:
# ⏭️ OPTIONAL — only for Vietnamese narration (VieNeu-TTS). Safe to skip.
import os, shutil, subprocess, sys

env = os.environ.copy()
cargo_bin = os.path.expanduser('~/.cargo/bin')

# 1. Rust toolchain for sea-g2p (no Linux x86_64 wheel on PyPI -> builds via maturin)
if shutil.which('cargo') is None and not os.path.exists(f'{cargo_bin}/cargo'):
    subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal',
                   shell=True, check=True)
env['PATH'] = f"{cargo_bin}:{env['PATH']}"

# 2. VieNeu runtime deps (llama-cpp-python compiles from source: ~10 min)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(PIXELLE_ROOT / 'requirements-vieneu.txt')], check=True, env=env)

# 3. vieneu itself WITHOUT its declared deps: they would clobber the Wan2GP
#    environment from step 5 (gradio>=5.49.1 vs 5.29.0, CPU onnxruntime vs
#    onnxruntime-gpu). Everything it actually needs is already installed.
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'vieneu==2.7.0', '--no-deps'], check=True, env=env)

print("✅ VieNeu-TTS installed — set TTS_MODE = 'vieneu' in step 6 "
      "(voices: Xuân Vĩnh, Phạm Tuyên, Bích Ngọc, Thục Đoan — fuzzy-matched).")


## 6. Configure Pixelle-Video

Fill in your **LLM endpoint** (required) and adjust the media workflows / TTS voice if you like, then run the cell — it writes `Pixelle_video/config.yaml` (shared by all the apps).

The `wan2gp/...` workflows are *descriptors* that map to WanGP models (see `Pixelle_video/WAN2GP_BACKEND.md`). Model checkpoints are downloaded automatically by WanGP on first use.


In [ ]:
import yaml

# --- LLM (required: analyzes the PDF, writes the script & media prompts) ----
LLM_API_KEY  = ''                                  # <-- your API key
LLM_BASE_URL = 'https://api.deepseek.com'          # any OpenAI-compatible endpoint
LLM_MODEL    = 'deepseek-chat'

# --- Media generation (wan2gp = models loaded in-process by WanGP) ---------
IMAGE_WORKFLOW = 'wan2gp/image_z_image.json'       # Z-Image Turbo 6B  (T4-friendly)
VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_1.3B.json'   # Wan 2.1 1.3B t2v  (T4-friendly)
# Bigger GPUs:
#   IMAGE_WORKFLOW = 'wan2gp/image_qwen.json'             # Qwen Image 20B
#   VIDEO_WORKFLOW = 'wan2gp/video_wan2.1_fusionx.json'   # Wan 2.1 FusioniX 14B
#   VIDEO_WORKFLOW = 'wan2gp/video_ltx2_distilled.json'   # LTX-2 22B (video + audio)

PROMPT_PREFIX = ('Minimalist black-and-white matchstick figure style illustration, '
                 'clean lines, simple sketch style')
# Tip: for PDF videos you can leave PROMPT_PREFIX = '' and let the digest's
# "visual world" carry the style — or keep a prefix to force a fixed look.

# --- TTS ---------------------------------------------------------------------
TTS_MODE  = 'local'              # 'local' = edge-tts (online, free)
                                 # 'vieneu' = Vietnamese on-device (run cell 5c first!)
TTS_VOICE = 'en-US-GuyNeural'    # local: e.g. 'zh-CN-YunjianNeural' for Chinese
                                 # vieneu: 'Xuân Vĩnh', 'Phạm Tuyên', 'Bích Ngọc', 'Thục Đoan' (fuzzy-matched)
TTS_SPEED = 1.2                  # vieneu tip: 1.0 sounds most natural
TTS_NORMALIZE = True             # loudness-normalize the narration to -16 LUFS (fixes quiet TTS)
TTS_VOLUME = 1.0                 # extra gain on top (1.0 = unchanged, 1.5 = +50%; clip-guarded)

# --- WanGP runtime -----------------------------------------------------------
WAN2GP_CLI_ARGS = ['--profile', '5']   # low-VRAM profile (same as the official notebook)

config = {
    'project_name': 'Pixelle-Video',
    'llm': {
        'api_key': LLM_API_KEY,
        'base_url': LLM_BASE_URL,
        'model': LLM_MODEL,
        'enable_thinking': False,
    },
    'comfyui': {
        'comfyui_url': 'http://127.0.0.1:8188',
        'comfyui_api_key': None,
        'runninghub_api_key': None,
        'runninghub_concurrent_limit': 1,
        'runninghub_instance_type': None,
        'tts': {
            'inference_mode': TTS_MODE,
            'local': {'voice': TTS_VOICE if TTS_MODE == 'local' else 'zh-CN-YunjianNeural',
                      'speed': TTS_SPEED if TTS_MODE == 'local' else 1.2},
            'vieneu': {'voice': TTS_VOICE if TTS_MODE == 'vieneu' else 'Xuân Vĩnh (Nam - Miền Nam)',
                       'speed': TTS_SPEED if TTS_MODE == 'vieneu' else 1.0,
                       'ref_audio': None, 'mode': 'turbo', 'device': 'cpu'},
            'comfyui': {'default_workflow': None},
        },
        'image': {'default_workflow': IMAGE_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
        'video': {'default_workflow': VIDEO_WORKFLOW, 'prompt_prefix': PROMPT_PREFIX},
    },
    'wan2gp': {
        'root': str(WAN2GP_ROOT),
        'cli_args': WAN2GP_CLI_ARGS,
        'output_dir': None,
    },
    'template': {'default_template': '1080x1920/image_default.html'},
}

(PIXELLE_ROOT / 'config.yaml').write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print('config.yaml written:')
print((PIXELLE_ROOT / 'config.yaml').read_text())


## 7. Initialize the core + the PDF → Video engine

Sets up paths, initializes the core services and creates the `PdfVideoEngine` (it extends the scene-by-scene engine, so all per-scene generation steps are shared). The WanGP session itself is created lazily — model weights are only loaded (and downloaded) on the first generation.


In [ ]:
import os, sys

os.chdir(PIXELLE_ROOT)                                   # relative paths: workflows/, templates/, output/
os.environ['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)

# sys.path: PDF app (pdfv) + scene-by-scene engine (sbs) + core + Wan2GP root
for p in (str(WAN2GP_ROOT), str(PIXELLE_ROOT), str(SBS_ROOT), str(PDF_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

from pixelle_video.service import pixelle_video
from pdfv import PdfVideoEngine

await pixelle_video.initialize()
engine = PdfVideoEngine(pixelle_video)
print(pixelle_video)
print('wan2gp media workflows:', [w for w in pixelle_video.media.available if w.startswith('wan2gp/')])


## 8. Get a PDF

Three ways to provide the document, in priority order:
1. set `PDF_PATH` to a file already in the runtime (e.g. from Google Drive),
2. leave it empty and **upload** a file when prompted (Colab),
3. do nothing — a small sample paper (*Attention Is All You Need*) is downloaded so you can try the pipeline immediately.


In [ ]:
import os, urllib.request

PDF_PATH = ''                      # <-- path to your PDF, or leave empty to upload

if not PDF_PATH:
    try:
        from google.colab import files   # only exists on Colab
        print('Select a PDF to upload (or press Cancel to use the sample paper)...')
        uploaded = files.upload()
        if uploaded:
            PDF_PATH = list(uploaded)[0]
    except ImportError:
        pass

if not PDF_PATH:
    PDF_PATH = str(WAN2GP_ROOT / 'sample_attention_is_all_you_need.pdf')
    if not os.path.exists(PDF_PATH):
        print('No PDF provided — downloading the sample paper...')
        urllib.request.urlretrieve('https://arxiv.org/pdf/1706.03762', PDF_PATH)

print(f'PDF: {os.path.abspath(PDF_PATH)} ({os.path.getsize(PDF_PATH) / 1e6:.1f} MB)')


## 9. Step ① — Ingest the PDF

Extracts and cleans the text (running headers/footers dropped, hyphenation repaired), keeps the metadata and table of contents. **No LLM call yet** — review what was extracted before spending tokens.

> Use `PAGE_RANGE` to video only a part of a long document (a chapter, the results section, ...). Page numbers are 1-based and inclusive.


In [ ]:
PAGE_RANGE = None          # e.g. (1, 12) to use only those pages

doc = engine.ingest_pdf(PDF_PATH, page_range=PAGE_RANGE)
print(doc.describe())

if doc.toc:
    print()
    print('Table of contents (first entries):')
    for level, toc_title, page in doc.toc[:12]:
        print(f"  {'  ' * (level - 1)}- {toc_title}  (p.{page})")

print()
print('--- First page excerpt ' + '-' * 40)
print(doc.pages[0].text[:600])


## 10. Step ② — Digest the document (then review it)

The map-reduce analysis: long documents are chunked and each chunk reduced to structured notes (key points + faithful facts/quotes); one final pass builds the **video digest** — core message, grounded key insights, hook ideas, the shared **visual world**, tone and language.

> `FOCUS` steers the whole video (e.g. `'focus on the experimental results'`, `'chỉ tập trung vào chương 2'`). It is reused by the script step below.


In [ ]:
FOCUS = None               # e.g. 'focus on the experimental results'

def on_digest_progress(done, total, message):
    print(f'  [{done}/{total}] {message}')

digest = await engine.digest_document(doc, focus=FOCUS, progress_callback=on_digest_progress)
print()
print(digest.describe())


### Edit the digest (optional)

The digest drives everything downstream — tune it before scripting:


In [ ]:
# --- Tune the digest before scripting (uncomment & adapt) --------------------
# digest.visual_world = 'a softly lit paper-craft world folded from book pages, warm amber light'
# digest.tone         = 'curious and encouraging'
# digest.core_message = 'My sharper one-line takeaway.'
# del digest.key_insights[3]            # drop an insight you don't want in the video

print(f'Visual world: {digest.visual_world}')
print(f'Tone:         {digest.tone}')
print(f'Core message: {digest.core_message}')
print(f'Insights:     {len(digest.key_insights)}')


## 11. Step ③ — Generate the script (then review it)

The LLM writes the video title plus one narration per scene: **hook → one grounded insight per scene → takeaway**. Every fact must come from the digest.

> Template naming controls the media type: `image_*.html` templates use the image model, `video_*.html` templates use the video model, `static_*.html` use no media model at all.
>
> `NARRATION_LANGUAGE = None` keeps the document's own language; set e.g. `'Vietnamese'` to narrate an English paper in Vietnamese (pair with the VieNeu voice from cell 5c).


In [ ]:
N_SCENES = 5
TEMPLATE = '1080x1920/image_default.html'   # try '1080x1920/video_default.html' for AI video scenes
NARRATION_LANGUAGE = None                   # None = the document's language; or 'Vietnamese', 'English', ...

from pixelle_video.utils.template_util import resolve_template_path, get_template_type
from pixelle_video.services.frame_html import HTMLFrameGenerator

template_type = get_template_type(TEMPLATE.split('/')[-1])    # 'static' | 'image' | 'video'
media_workflow = {'image': IMAGE_WORKFLOW, 'video': VIDEO_WORKFLOW}.get(template_type)
media_width, media_height = HTMLFrameGenerator(resolve_template_path(TEMPLATE)).get_media_size()
print(f'Template type: {template_type} | media workflow: {media_workflow} | media size: {media_width}x{media_height}')
print()

title, narrations = await engine.generate_pdf_script(
    digest,
    n_scenes=N_SCENES,
    focus=FOCUS,
    language=NARRATION_LANGUAGE,
)

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


### Edit the script (optional)

Adjust anything before moving on:

- **manual edit**: assign directly into the `narrations` list (or change `title`)
- **✨ AI rewrite**: ask the LLM to rewrite one scene, optionally with an instruction
- **add / remove scenes**: regular Python list operations


In [ ]:
# --- Manual edits (uncomment & adapt) ---------------------------------------
# title = 'My better title'
# narrations[0] = 'My own opening line for scene 1.'
# narrations.append('One extra closing scene.')
# del narrations[2]

# --- AI rewrite of a single scene (uncomment to use) -------------------------
# narrations[0] = await engine.rewrite_narration(
#     narrations[0], topic=digest.core_message,
#     instruction='make it a question that hooks the viewer',
# )

print(f'Title: {title}')
for i, n in enumerate(narrations, 1):
    print(f'  Scene {i}: {n}')


## 12. Step ④ — Generate the visual prompts (then review them)

One **English** media prompt per scene, all living inside the digest's visual world (same setting/palette, different composition per scene), with the style `PROMPT_PREFIX` already applied. Skipped automatically for `static_*` templates.


In [ ]:
if template_type == 'static':
    prompts = [None] * len(narrations)
    print('Static template — no media prompts needed.')
else:
    prompts = await engine.generate_visual_prompts(narrations, digest, prompt_prefix=PROMPT_PREFIX)
    for i, p in enumerate(prompts, 1):
        print(f'Scene {i}: {p}\n')


In [ ]:
# --- Edit prompts before generation (uncomment & adapt) ----------------------
# prompts[0] = 'wide shot of ' + digest.visual_world + ', a lone figure pausing at a fork, ' + PROMPT_PREFIX

# --- Or let the AI regenerate a single one (stays in the visual world) --------
# prompts[1] = await engine.generate_visual_prompt_for(narrations[1], digest, prompt_prefix=PROMPT_PREFIX)

for i, p in enumerate(prompts, 1):
    print(f'Scene {i}: {p}\n')


## 13. Create the project

Freezes the script + prompts + settings into a `SceneProject` with its own task directory (`Pixelle_video/output/<task_id>/`). Each scene gets a stable `uid` so its assets can be regenerated safely. The PDF provenance (file, pages, focus, visual world) is kept in the task metadata.


In [ ]:
params = {
    'text': f'PDF: {digest.title}',
    'mode': 'pdf',
    'n_scenes': len(narrations),
    'split_mode': 'paragraph',
    'title': title,
    'tts_inference_mode': TTS_MODE,
    'tts_voice': TTS_VOICE,
    'tts_speed': TTS_SPEED,
    'tts_normalize': TTS_NORMALIZE,   # post-TTS loudness normalization (engine-applied)
    'tts_volume': TTS_VOLUME,         # extra narration gain on top
    'frame_template': TEMPLATE,
    'template_params': None,
    'media_workflow': media_workflow,
    'prompt_prefix': PROMPT_PREFIX,
    'media_width': media_width,
    'media_height': media_height,
    # --- PDF provenance (persisted with the task) ---
    'pdf_source': doc.path,
    'pdf_pages': list(doc.page_range),
    'pdf_focus': FOCUS,
    'pdf_language': NARRATION_LANGUAGE or digest.language,
    'pdf_visual_world': digest.visual_world,
}

project = engine.create_project(title, narrations, prompts, params)
print(f'Task: {project.task_id}  ({len(project.scenes)} scenes, media={project.media_requirement})')
print(f'Dir:  {project.task_dir}')


## 14. Step ⑤ — Generate scene 1 piece by piece

Each sub-step produces a previewable output. Re-run any cell to regenerate just that piece.

> The **first** media generation downloads the model checkpoint (a few GB) — subsequent scenes reuse the model already loaded in VRAM.


In [ ]:
# 🎤 Audio (TTS) — the narration voiceover; its duration drives video-clip length
from IPython.display import Audio, display

scene = project.scenes[0]
await engine.generate_audio(project, scene, 0)
print(f'Duration: {scene.duration:.2f}s')
display(Audio(scene.audio_path))


In [ ]:
# 🖼️/🎬 Media (WanGP, in-process) — skipped for static templates
from IPython.display import Image, Video, display

if project.needs_media:
    await engine.generate_media(project, scene, 0)
    if scene.media_type == 'video':
        display(Video(scene.video_path, embed=True, width=300))
    else:
        display(Image(scene.image_path, width=300))
else:
    print('Static template — no media for this scene.')


In [ ]:
# 🎞️ Segment — subtitled HTML frame + audio, the final building block of the video
from IPython.display import Video, display

await engine.render_segment(project, scene, 0)
display(Video(scene.segment_path, embed=True, width=300))


### Not happy with scene 1? Regenerate any piece

Edit the text and re-run only what changed — invalidation is automatic:


In [ ]:
# --- Change the narration (invalidates audio + segment) ----------------------
# scene.narration = 'A brand new opening line.'
# scene.invalidate_audio()
# await engine.generate_audio(project, scene, 0)

# --- Change the prompt (invalidates media + segment) --------------------------
# scene.prompt = await engine.generate_visual_prompt_for(scene.narration, digest, prompt_prefix=PROMPT_PREFIX)
# scene.invalidate_media()
# await engine.generate_media(project, scene, 0)

# --- Then re-render the segment -----------------------------------------------
# await engine.render_segment(project, scene, 0)

print('Scene 1 status:',
      'audio ✅' if scene.audio_path else 'audio ⬜',
      '| media ✅' if scene.has_media else '| media ⬜',
      '| segment ✅' if scene.segment_path else '| segment ⬜')


## 15. Generate the remaining scenes

`process_scene` runs whatever is still missing for each scene (audio → media → segment). Re-run this cell safely — finished scenes are skipped.


In [ ]:
for i, sc in enumerate(project.scenes):
    if sc.segment_path:
        print(f'Scene {i+1}: already done ✅')
        continue
    await engine.process_scene(
        project, sc, i,
        progress_callback=lambda stage, i=i: print(f'  Scene {i+1}: {stage}...'),
    )
    print(f'Scene {i+1}: done ✅ ({sc.duration:.1f}s)')

print()
print(f'Segments ready: {sum(1 for s in project.scenes if s.segment_path)}/{len(project.scenes)}')


### Review all segments (optional)


In [ ]:
from IPython.display import Video, display

for i, sc in enumerate(project.scenes, 1):
    print(f'Scene {i}: {sc.narration}')
    display(Video(sc.segment_path, embed=True, width=260))


## 16. Step ⑥ — Compose the final video

Concatenates all segments and (optionally) adds background music, then persists the task (labeled `pdf_to_video`) so it appears in the web UI's History page.


In [ ]:
BGM = None            # or e.g. 'default.mp3' (any file in Pixelle_video/bgm/)
BGM_VOLUME = 0.2

result = await engine.compose_final(project, bgm_path=BGM, bgm_volume=BGM_VOLUME)

print(f"Final video: {result['video_path']}")
print(f"Duration:    {result['duration']:.1f}s | Size: {result['file_size'] / 1e6:.1f} MB | Scenes: {result['n_scenes']}")


### Preview the result


In [ ]:
from IPython.display import Video

Video(result['video_path'], embed=True, width=320)


## 17. (Optional) Launch the PDF → Video Web UI

The same flow as a 6-step **Streamlit wizard** (Setup/PDF upload → Digest → Script → Prompts → Scenes → Final), exposed through a free Cloudflare quick tunnel. Click the printed `trycloudflare.com` link; keep the cell running while you use the UI and press **Stop** when done.

It shares the same config / output as this notebook, so videos composed in the UI also land in `Pixelle_video/output/` and show up in the History page.

> **If the page errors with `Failed to fetch dynamically imported module`:** do a **hard refresh** (`Ctrl+Shift+R` / `Cmd+Shift+R`) — the browser cached the UI of a previous run whose JS chunks no longer exist. If it persists, stop and re-run this cell to get a fresh tunnel (the free tunnel occasionally drops chunk requests; the cell already forces the more reliable http2 transport).


In [ ]:
import os, re, subprocess, sys

# Cloudflare quick tunnel binary
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

env = os.environ.copy()
env['PIXELLE_VIDEO_ROOT'] = str(PIXELLE_ROOT)
env['PYTHONPATH'] = f"{PDF_ROOT}:{SBS_ROOT}:{WAN2GP_ROOT}:{PIXELLE_ROOT}"

streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(PDF_ROOT / 'web' / 'app.py'),
     '--server.port', '8503', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
     '--server.maxUploadSize', '200',
     '--browser.gatherUsageStats', 'false'],
    cwd=str(PIXELLE_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8503',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print('Waiting for the tunnel URL...')
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n🌐 PDF → Video Web UI: {match.group(0)}\n')
        break

try:
    for line in iter(streamlit_proc.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping...')
finally:
    streamlit_proc.terminate()
    tunnel_proc.terminate()
    print('Web UI and tunnel stopped.')


## Notes & troubleshooting

- **Outputs** land in `Pixelle_video/output/<task_id>/final.mp4`; per-scene assets in `frames/<scene_uid>_*.{mp3,png,mp4}` (uid-based, so regenerating never collides).
- **Scanned PDFs**: if ingestion reports almost no text, the PDF has no text layer — run OCR first (`ocrmypdf in.pdf out.pdf`) and ingest the result.
- **Very large PDFs**: `digest_document` analyzes at most `max_chunks=12` chunks (~144k chars), evenly sampled with the first and last always kept. Raise `max_chunks`, or better, set `PAGE_RANGE` to the part you actually want to video.
- **Wrong facts in the script?** The script may only use facts from the digest — check the digest's `grounding` fields first (step 10); fix the digest, then regenerate the script.
- **Other-language narration**: set `NARRATION_LANGUAGE` in step 11 (e.g. `'Vietnamese'` + the VieNeu voice from cell 5c) to narrate a document in a different language than it is written in.
- **Narration too quiet?** The engine loudness-normalizes every narration to -16 LUFS after TTS (`tts_normalize`, on by default) and can add extra gain (`tts_volume`, e.g. 1.5) — both set in steps 6/13. Alternatively lower the BGM (`BGM_VOLUME`) when composing.
- **Visual style**: the digest's `visual_world` carries scene-to-scene coherence; `PROMPT_PREFIX` (step 6) forces a fixed rendering style on top. Use either or both.
- **Regeneration rules**: changing a narration invalidates that scene's audio + segment; changing a prompt invalidates its media + segment. `process_scene` only re-runs what's missing.
- **First generation is slow**: WanGP downloads the model checkpoint, then keeps it loaded in VRAM — later scenes are much faster.
- **Out of VRAM / RAM on T4**: stick to `image_z_image` + `video_wan2.1_1.3B`, keep `--profile 5`, and use smaller media sizes (the template's media size is capped automatically by each descriptor's `max_pixels`).
- **Reasoning LLMs** (MiniMax-M3, DeepSeek-R1, Qwen3, ...): hidden chain-of-thought counts against the token budget, which can yield `LLM returned no content`. The PDF stages (digest / script / visual prompts) retry automatically with an escalating budget (up to 24k tokens for the script); if a step still fails, switch to a non-reasoning model (e.g. `deepseek-chat`) for a much faster pipeline.
- **`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'`**: an old system `pkg_resources` is shadowing the modern one on Python 3.12. Run `pip install --upgrade setuptools wheel` and restart (the install cell now does this automatically).
- Full backend documentation: `Pixelle_video/WAN2GP_BACKEND.md` · app documentation: `Pixelle_video_pdf/README.md` (the web UI lives in `Pixelle_video_pdf/web/`, port 8503) · per-scene engine: `Pixelle_video_scene_by_scene/README.md`.
